# Insurance Agent Payroll & Commission Calculator

**Objective:** To build a rule-based Python calculator that processes monthly sales data for insurance agents and generates a payslip. 

**Business Rules:**
* Agents receive a base monthly salary.
* Commission is volume-based and varies by product type (Life, Health, Auto).
* Net pay is calculated using 2026/2027 Scottish Income Tax bands.

In [1]:
# --- SYSTEM CONFIGURATION ---

# 2026/2027 Scottish Income Tax Bands (Yearly)
# For simplicity in this project, we will divide the yearly thresholds by 12 for monthly calculations
tax_brackets_yearly = {
    "personal_allowance": 12570,
    "starter_rate": {"rate": 0.19, "threshold_max": 16537},      
    "basic_rate": {"rate": 0.20, "threshold_max": 29526},        
    "intermediate_rate": {"rate": 0.21, "threshold_max": 43662}, 
}

# Commission payouts per policy sold (£)
commission_rates = {
    "life_insurance": 150.00,
    "health_insurance": 50.00,
    "home_auto_insurance": 20.00
}

print("System Configuration Loaded Successfully.")

System Configuration Loaded Successfully.


In [2]:
# --- MONTHLY INPUT DATA ---

agent_data = {
    "agent_id": "AG-001",
    "name": "Jane Doe",
    "base_salary_monthly": 2000.00,
    "sales_this_month": {
        "life_insurance": 4,          # High value, low volume
        "health_insurance": 12,       # Medium value
        "home_auto_insurance": 25     # Low value, high volume
    }
}

print(f"Loaded data for Agent: {agent_data['name']}")

Loaded data for Agent: Jane Doe


In [3]:
# --- FUNCTION: CALCULATE COMMISSION ---

def calculate_commission(sales_data, rates):
    """
    Calculates total commission based on sales volume and product payout rates.
    """
    total_commission = 0.0
    
    print("--- Commission Breakdown ---")
    
    # Loop through each product the agent sold
    for product, quantity in sales_data.items():
        # Look up the payout rate for this specific product
        # We use .get() so if a product isn't found, it safely defaults to £0
        rate = rates.get(product, 0)
        
        # Calculate commission for this specific product line
        product_commission = quantity * rate
        
        # Add it to our running total
        total_commission += product_commission
        
        # Print a formatted string to show the math
        formatted_name = product.replace('_', ' ').title()
        print(f"{formatted_name}: {quantity} sold at £{rate:.2f} = £{product_commission:.2f}")
        
    return total_commission

# --- TEST THE FUNCTION ---

# We pass the specific data dictionaries we created in previous cells into our new function
monthly_commission = calculate_commission(agent_data["sales_this_month"], commission_rates)

print("-" * 28)
print(f"Total Commission Earned: £{monthly_commission:.2f}")

--- Commission Breakdown ---
Life Insurance: 4 sold at £150.00 = £600.00
Health Insurance: 12 sold at £50.00 = £600.00
Home Auto Insurance: 25 sold at £20.00 = £500.00
----------------------------
Total Commission Earned: £1700.00


In [4]:
# --- FUNCTION: CALCULATE DEDUCTIONS & NET PAY ---

def calculate_net_pay(monthly_gross, tax_brackets):
    """
    Calculates monthly tax and National Insurance (NI) based on annualized salary,
    using progressive tax bands.
    """
    # 1. Annualize the salary for accurate bracket calculation
    yearly_gross = monthly_gross * 12
    yearly_tax = 0.0
    
    # Extract the personal allowance (tax-free amount)
    allowance = tax_brackets["personal_allowance"]
    
    # 2. Calculate Progressive Income Tax
    if yearly_gross > allowance:
        taxable_income = yearly_gross - allowance
        
        # Starter Rate (19%)
        starter_max = tax_brackets["starter_rate"]["threshold_max"] - allowance
        if taxable_income > 0:
            taxed_amount = min(taxable_income, starter_max)
            yearly_tax += taxed_amount * tax_brackets["starter_rate"]["rate"]
            taxable_income -= taxed_amount
            
        # Basic Rate (20%)
        basic_max = tax_brackets["basic_rate"]["threshold_max"] - tax_brackets["starter_rate"]["threshold_max"]
        if taxable_income > 0:
            taxed_amount = min(taxable_income, basic_max)
            yearly_tax += taxed_amount * tax_brackets["basic_rate"]["rate"]
            taxable_income -= taxed_amount
            
        # Intermediate Rate (21%)
        intermediate_max = tax_brackets["intermediate_rate"]["threshold_max"] - tax_brackets["basic_rate"]["threshold_max"]
        if taxable_income > 0:
            taxed_amount = min(taxable_income, intermediate_max)
            yearly_tax += taxed_amount * tax_brackets["intermediate_rate"]["rate"]
            taxable_income -= taxed_amount
            
        # (For simplicity in this portfolio piece, we stop at the intermediate band)

    # 3. Calculate National Insurance (simplified 8% above allowance for 2026/2027)
    yearly_ni = 0.0
    if yearly_gross > allowance:
        yearly_ni = (yearly_gross - allowance) * 0.08

    # 4. Convert back to monthly figures
    monthly_tax = yearly_tax / 12
    monthly_ni = yearly_ni / 12
    net_pay = monthly_gross - monthly_tax - monthly_ni
    
    return monthly_tax, monthly_ni, net_pay

# --- TEST THE FUNCTION ---

# Calculate the gross pay using our previous variable
gross_pay = agent_data["base_salary_monthly"] + monthly_commission

# Run our new function
tax, ni, net = calculate_net_pay(gross_pay, tax_brackets_yearly)

print("--- Deductions Breakdown ---")
print(f"Gross Pay: £{gross_pay:.2f}")
print(f"Income Tax: £{tax:.2f}")
print(f"National Insurance: £{ni:.2f}")
print("-" * 28)
print(f"Net Take-Home Pay: £{net:.2f}")

--- Deductions Breakdown ---
Gross Pay: £3700.00
Income Tax: £526.67
National Insurance: £212.20
----------------------------
Net Take-Home Pay: £2961.13


In [5]:
# --- FUNCTION: GENERATE PAYSLIP ---

def generate_payslip(agent_dict, gross, comm, tax_deduct, ni_deduct, net):
    """
    Generates a formatted text-based payslip displaying all required UK components.
    """
    print("=" * 40)
    print("         OFFICIAL UK PAYSLIP")
    print("=" * 40)
    print(f"Employee Name:   {agent_dict['name']}")
    print(f"Payroll Number:  {agent_dict['agent_id']}")
    print(f"Tax Code:        S1257L") 
    print("-" * 40)
    
    print("EARNINGS")
    print(f"Base Salary:     £{agent_dict['base_salary_monthly']:>9.2f}")
    print(f"Commission:      £{comm:>9.2f}")
    print(f"Gross Pay:       £{gross:>9.2f}")
    print("-" * 40)
    
    print("DEDUCTIONS")
    print(f"Income Tax:      £{tax_deduct:>9.2f}")
    print(f"Nat. Insurance:  £{ni_deduct:>9.2f}")
    print("-" * 40)
    
    # \033[1m and \033[0m add bold text formatting in the terminal/Jupyter
    print(f"\033[1mNET TAKE-HOME PAY: £{net:>9.2f}\033[0m")
    print("=" * 40)

# --- RUN THE GENERATOR ---

# We pass in the data dictionary and all the variables we saved from our previous functions
generate_payslip(agent_data, gross_pay, monthly_commission, tax, ni, net)

         OFFICIAL UK PAYSLIP
Employee Name:   Jane Doe
Payroll Number:  AG-001
Tax Code:        S1257L
----------------------------------------
EARNINGS
Base Salary:     £  2000.00
Commission:      £  1700.00
Gross Pay:       £  3700.00
----------------------------------------
DEDUCTIONS
Income Tax:      £   526.67
Nat. Insurance:  £   212.20
----------------------------------------
NET TAKE-HOME PAY: £  2961.13
